# 02. 모델 학습 및 평가

## 데이터 불러오기

이 노트북을 독립적으로 실행할 수 있도록
원본 측정값(X)과 정상·불량 정답(y)을 다시 읽는다.

In [2]:
from pathlib import Path
import pandas as pd

project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

data_dir = project_dir / "data" / "raw"

# 공정 측정값
X = pd.read_csv(
    data_dir / "secom.data",
    sep=r"\s+",
    header=None
)

# 검사 결과와 검사 시각
labels = pd.read_csv(
    data_dir / "secom_labels.data",
    sep=r"\s+",
    header=None,
    names=["result", "timestamp"]
)

# 정상 0, 불량 1로 변환
y = labels["result"].map({-1: 0, 1: 1})
y.name = "is_defect"

print("측정값 크기:", X.shape)
print("정답 개수:", len(y))

측정값 크기: (1567, 590)
정답 개수: 1567


## 8. 학습·검증·테스트 데이터 분리

데이터를 약 60% / 20% / 20%로 나누고,
각 그룹의 정상·불량 비율이 비슷하게 유지되도록 한다.
결측치 처리와 측정 항목 선택 기준은 학습 데이터에서 정한다.

In [3]:
from sklearn.model_selection import train_test_split

# 전체 데이터의 20%를 테스트용으로 분리
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# 남은 80%의 25%를 검증용으로 분리
# 전체 기준으로 학습 60%, 검증 20%, 테스트 20%
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,
    stratify=y_train_val,
    random_state=42
)

split_summary = pd.DataFrame({
    "데이터 수": [len(y_train), len(y_val), len(y_test)],
    "정상 수": [
        (y_train == 0).sum(),
        (y_val == 0).sum(),
        (y_test == 0).sum()
    ],
    "불량 수": [
        y_train.sum(),
        y_val.sum(),
        y_test.sum()
    ],
    "불량 비율(%)": [
        y_train.mean() * 100,
        y_val.mean() * 100,
        y_test.mean() * 100
    ]
}, index=["학습", "검증", "테스트"])

split_summary.round(2)

,데이터 수,정상 수,불량 수,불량 비율(%)
학습,939,877,62,6.60
검증,314,293,21,6.69
테스트,314,293,21,6.69


## 9. 사용할 측정 항목 선택

학습 데이터에서 결측치 비율이 50%를 초과하거나
관측된 값이 한 종류 이하인 열을 제외한다.
선택한 열을 검증·테스트 데이터에도 동일하게 적용한다.

In [4]:
# 학습 데이터에서만 제외 기준 계산
train_missing_ratio = X_train.isna().mean()
train_unique_count = X_train.nunique(dropna=True)

drop_mask = (
    (train_missing_ratio > 0.5)
    | (train_unique_count <= 1)
)

selected_columns = X_train.columns[~drop_mask]

# 같은 측정 항목을 세 그룹에 적용
X_train_selected = X_train.loc[:, selected_columns].copy()
X_val_selected = X_val.loc[:, selected_columns].copy()
X_test_selected = X_test.loc[:, selected_columns].copy()

print("기존 측정 항목 수:", X_train.shape[1])
print("제외한 항목 수:", drop_mask.sum())
print("남은 항목 수:", len(selected_columns))

기존 측정 항목 수: 590
제외한 항목 수: 146
남은 항목 수: 444


## 10. 결측치 채우기

학습 데이터에서 각 측정 항목의 중앙값을 계산해 결측치를 채운다.
검증·테스트 데이터에도 학습 데이터에서 계산한 중앙값을 적용한다.

In [5]:
from sklearn.impute import SimpleImputer
import numpy as np

imputer = SimpleImputer(strategy="median")

# 학습 데이터에서 중앙값 계산 후 결측치 채우기
X_train_imputed = imputer.fit_transform(X_train_selected)

# 학습 데이터의 중앙값을 그대로 적용
X_val_imputed = imputer.transform(X_val_selected)
X_test_imputed = imputer.transform(X_test_selected)

print("학습 데이터 크기:", X_train_imputed.shape)
print("학습 결측치 수:", np.isnan(X_train_imputed).sum())
print("검증 결측치 수:", np.isnan(X_val_imputed).sum())
print("테스트 결측치 수:", np.isnan(X_test_imputed).sum())

학습 데이터 크기: (939, 444)
학습 결측치 수: 0
검증 결측치 수: 0
테스트 결측치 수: 0


## 11. 측정값 표준화

로지스틱 회귀 학습을 위해 측정 항목별 숫자 규모를 맞춘다.
학습 데이터의 평균과 표준편차를 기준으로
학습·검증·테스트 데이터를 동일하게 변환한다.

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# 학습 데이터에서 기준 계산 후 표준화
X_train_scaled = scaler.fit_transform(X_train_imputed)

# 같은 기준으로 검증·테스트 데이터 변환
X_val_scaled = scaler.transform(X_val_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("학습 데이터 크기:", X_train_scaled.shape)
print("검증 데이터 크기:", X_val_scaled.shape)
print("테스트 데이터 크기:", X_test_scaled.shape)

print("첫 번째 항목의 학습 평균:",
      round(X_train_scaled[:, 0].mean(), 6))
print("첫 번째 항목의 학습 표준편차:",
      round(X_train_scaled[:, 0].std(), 6))

학습 데이터 크기: (939, 444)
검증 데이터 크기: (314, 444)
테스트 데이터 크기: (314, 444)
첫 번째 항목의 학습 평균: 0.0
첫 번째 항목의 학습 표준편차: 1.0


## 12. 로지스틱 회귀 모델 학습

표준화한 학습 데이터와 정상·불량 정답으로 모델을 학습한다.
불량 데이터가 적으므로 `class_weight="balanced"`를 적용해
학습 과정에서 불량 사례에 더 큰 가중치를 부여한다.

In [7]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

# 측정값과 정답으로 모델 학습
logistic_model.fit(X_train_scaled, y_train)

print("로지스틱 회귀 학습 완료")
print("학습한 측정 항목 수:", logistic_model.n_features_in_)
print("정답 클래스:", logistic_model.classes_)

로지스틱 회귀 학습 완료
학습한 측정 항목 수: 444
정답 클래스: [0 1]


## 13. 로지스틱 회귀 검증

학습에 사용하지 않은 검증 데이터로 정상·불량을 예측한다.
정확도와 불량 기준 Precision·Recall·F1을 확인한다.
테스트 데이터는 최종 평가 단계까지 사용하지 않는다.

In [8]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# 검증 데이터의 정상·불량 예측
logistic_val_pred = logistic_model.predict(X_val_scaled)

# 불량(1)을 기준으로 성능 계산
logistic_val_metrics = {
    "정확도": accuracy_score(y_val, logistic_val_pred),
    "불량 Precision": precision_score(
        y_val, logistic_val_pred, zero_division=0
    ),
    "불량 Recall": recall_score(
        y_val, logistic_val_pred, zero_division=0
    ),
    "불량 F1": f1_score(
        y_val, logistic_val_pred, zero_division=0
    )
}

pd.DataFrame(
    [logistic_val_metrics],
    index=["로지스틱 회귀"]
).round(4)

,정확도,불량 Precision,불량 Recall,불량 F1
로지스틱 회귀,0.8662,0.1613,0.2381,0.1923


## 14. Random Forest 학습 및 검증

결측치를 채운 학습 데이터로 Random Forest를 학습한다.
트리 모델은 표준화가 필수적이지 않아 표준화 전 데이터를 사용한다.
같은 검증 데이터와 평가 지표로 로지스틱 회귀와 비교한다.

In [9]:
from sklearn.ensemble import RandomForestClassifier

forest_model = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# 결측치를 채운 데이터로 학습
forest_model.fit(X_train_imputed, y_train)

# 검증 데이터 예측
forest_val_pred = forest_model.predict(X_val_imputed)

forest_val_metrics = {
    "정확도": accuracy_score(y_val, forest_val_pred),
    "불량 Precision": precision_score(
        y_val, forest_val_pred, zero_division=0
    ),
    "불량 Recall": recall_score(
        y_val, forest_val_pred, zero_division=0
    ),
    "불량 F1": f1_score(
        y_val, forest_val_pred, zero_division=0
    )
}

# 두 모델의 검증 성능 비교
model_comparison = pd.DataFrame(
    [logistic_val_metrics, forest_val_metrics],
    index=["로지스틱 회귀", "Random Forest"]
)

model_comparison.round(4)

,정확도,불량 Precision,불량 Recall,불량 F1
로지스틱 회귀,0.8662,0.1613,0.2381,0.1923
Random Forest,0.9331,0.0000,0.0000,0.0000


## 15. 불량 판단 기준 비교

두 모델의 불량 예측 확률에 여러 판단 기준을 적용한다.
검증 데이터에서 불량 탐지 성능과 정상 오탐 건수를 비교한다.

In [10]:
from sklearn.metrics import confusion_matrix

# 각 모델이 계산한 불량(클래스 1) 확률
logistic_val_prob = logistic_model.predict_proba(X_val_scaled)[:, 1]
forest_val_prob = forest_model.predict_proba(X_val_imputed)[:, 1]

results = []

for model_name, probabilities in [
    ("로지스틱 회귀", logistic_val_prob),
    ("Random Forest", forest_val_prob)
]:
    for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:
        predictions = (probabilities >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_val, predictions, labels=[0, 1]
        ).ravel()

        results.append({
            "모델": model_name,
            "판단 기준": threshold,
            "불량 Precision": precision_score(
                y_val, predictions, zero_division=0
            ),
            "불량 Recall": recall_score(
                y_val, predictions, zero_division=0
            ),
            "불량 F1": f1_score(
                y_val, predictions, zero_division=0
            ),
            "불량 탐지": tp,
            "불량 놓침": fn,
            "정상 오탐": fp
        })

threshold_comparison = pd.DataFrame(results)
threshold_comparison.round(4)

,모델,판단 기준,불량 Precision,불량 Recall,불량 F1,불량 탐지,불량 놓침,정상 오탐
0,로지스틱 회귀,0.1,0.0877,0.2381,0.1282,5,16,52
1,로지스틱 회귀,0.2,0.1064,0.2381,0.1471,5,16,42
2,로지스틱 회귀,0.3,0.1190,0.2381,0.1587,5,16,37
3,로지스틱 회귀,0.4,0.1429,0.2381,0.1786,5,16,30
4,로지스틱 회귀,0.5,0.1613,0.2381,0.1923,5,16,26
5,Random Forest,0.1,0.0844,0.9048,0.1545,19,2,206
6,Random Forest,0.2,0.1266,0.4762,0.2000,10,11,69
7,Random Forest,0.3,0.2000,0.1905,0.1951,4,17,16
8,Random Forest,0.4,0.4000,0.0952,0.1538,2,19,3
9,Random Forest,0.5,0.0000,0.0000,0.0000,0,21,0


## 16. 모델 및 판단 기준 선택

비교한 설정 중 검증 데이터의 불량 F1이 가장 높은
Random Forest와 판단 기준 0.2를 선택한다.

불량 21건 중 10건을 탐지했지만 정상 69건을 오탐했다.
선택한 설정은 변경하지 않고 03 노트북에서 최종 테스트한다.

In [11]:
selected_model_name = "Random Forest"
selected_model = forest_model
selected_threshold = 0.2

# Random Forest는 표준화 전 데이터를 사용
selected_scaler = None

print("선택 모델:", selected_model_name)
print("불량 판단 기준:", selected_threshold)

선택 모델: Random Forest
불량 판단 기준: 0.2
